# PISA Data Downloader
---
If you're not familiar with PISA, it stands for the Programme for International Student Assessment. It's run by the OECD every three years and tests about 600,000 fifteen-year-olds across 80+ countries in math, reading, and science. It's one of the biggest educational assessments in the world, very similar in purpose to the U.S. **National Assessment of Educational Progress (NAEP)**.

The raw data file is a **2 GB SPSS file with 1,200+ columns**. Most of those are individual questionnaire responses we don't need. In this notebook, we'll extract just the variables that matter for predicting math scores, clean them up, and output a tidy CSV.

**What this notebook does:**
1. Downloads the PISA 2022 Student Questionnaire data from the OECD
2. Auto-detects the variable names (they can vary between releases)
3. Cleans and maps codes to readable labels
4. Saves a ready-to-use CSV file

## Imports

In [1]:
import pyreadstat
import pandas as pd
import numpy as np
import os
import zipfile
import urllib.request
from tqdm import tqdm

## Downloading the Data from OECD

We're pulling the **Student Questionnaire data file** directly from the OECD's PISA 2022 data repository. This is the same file that researchers and governments use worldwide.

It's about **800 MB compressed**. The cell checks if you've already downloaded it so you don't redownload every time you restart the notebook.

In [2]:
ZIP_URL = "https://webfs.oecd.org/pisa2022/STU_QQQ_SPSS.zip"
ZIP_PATH = "STU_QQQ_SPSS.zip"

if not os.path.exists(ZIP_PATH):
    print(f"Downloading PISA 2022 student data from OECD (~800 MB)...")
    print(f"Source: {ZIP_URL}")

    class _Progress(tqdm):
        def update_to(self, b=1, bsize=1, tsize=None):
            if tsize is not None:
                self.total = tsize
            self.update(b * bsize - self.n)

    with _Progress(unit="B", unit_scale=True, unit_divisor=1024, miniters=1) as t:
        urllib.request.urlretrieve(ZIP_URL, ZIP_PATH, reporthook=t.update_to)
    print(f"Download complete: {os.path.getsize(ZIP_PATH) / 1e6:.0f} MB")
else:
    print(f"Already downloaded: {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1e6:.0f} MB)")

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    all_files = z.namelist()
    sav_files = [f for f in all_files if f.lower().endswith('.sav')]
    print(f"\nFiles in archive: {all_files}")
    assert len(sav_files) > 0, "No .sav file found in the zip!"
    SAV_FILENAME = sav_files[0]
    
    if not os.path.exists(SAV_FILENAME):
        print(f"Extracting {SAV_FILENAME}...")
        z.extract(SAV_FILENAME)
        print(f"Extracted: {os.path.getsize(SAV_FILENAME) / 1e9:.1f} GB")
    else:
        print(f"Already extracted: {SAV_FILENAME} ({os.path.getsize(SAV_FILENAME) / 1e9:.1f} GB)")

print(f"\nData file ready: {SAV_FILENAME}")

Source: https://webfs.oecd.org/pisa2022/STU_QQQ_SPSS.zip


651MB [14:08, 804kB/s]                                


Download complete: 682 MB

Files in archive: ['CY08MSP_STU_QQQ.SAV']
Extracting CY08MSP_STU_QQQ.SAV...
Extracted: 2.1 GB

Data file ready: CY08MSP_STU_QQQ.SAV


## Quick Check Column Names

Before loading any actual data, we read **just the metadata**. 

Why do we need this step? Because PISA variable names can differ slightly between data releases. Instead of hardcoding names and hoping they match, we search for each variable we need and match it against what's actually in the file. This makes the notebook robust — it'll work even if OECD tweaks a column name in a future release.

Here are the variables we're looking for and why:

| Variable | What It Is | Use |
|---|---|---|
| `CNT` | Country 3-letter code | Categorical feature |
| `ST004D01T` | Student gender | Categorical feature |
| `MISCED` | Mother's education (ISCED scale) | Categorical feature |
| `FISCED` | Father's education (ISCED scale) | Categorical feature |
| `ESCS` | Economic, Social & Cultural Status index | Key continuous feature |
| `ST013Q01TA` | Books at home (1-6 scale) | Categorical feature |
| `ST011Q06TA` | Internet at home (1=Yes, 2=No) | Categorical feature |
| `MMINS` | Minutes per week studying math | Continuous feature |
| `CLSIZE` | Class size | Continuous feature |
| `PV1MATH` | Math plausible value #1 — **our target** | Target variable |

A quick note on **plausible values**: PISA doesn't give each student a single "score." It generates 10 plausible values (PV1MATH through PV10MATH) that represent the range of proficiency the student likely falls in. For this tutorial, we'll use PV1MATH. In a full research analysis, you'd run the model on all 10 and pool the results.

In [3]:
# Read only metadata — instant, no data loaded
_, meta = pyreadstat.read_sav(SAV_FILENAME, metadataonly=True)
all_cols = meta.column_names
print(f"File has {len(all_cols)} columns total.")

COLS = ['CNT', 'ST004D01T', 'MISCED', 'FISCED', 'ESCS',
        'HOMEPOS', 'ICTRES', 'STUDYHMW', 'PV1MATH']

found = [c for c in COLS if c in all_cols]
missing = [c for c in COLS if c not in all_cols]
print(f"\nFound {len(found)}/{len(COLS)} variables: {found}")
if missing:
    print(f"Missing: {missing}")

print(f"\nLoading {len(found)} columns...")
pdf, meta = pyreadstat.read_sav(SAV_FILENAME, usecols=found)
print(f"Loaded: {len(pdf):,} rows x {len(pdf.columns)} columns")
print(f"\nMissing values:")
print(pdf.isnull().sum())
pdf.head()

File has 1278 columns total.

Found 9/9 variables: ['CNT', 'ST004D01T', 'MISCED', 'FISCED', 'ESCS', 'HOMEPOS', 'ICTRES', 'STUDYHMW', 'PV1MATH']

Loading 9 columns...
Loaded: 613,744 rows x 9 columns

Missing values:
CNT               0
ST004D01T        79
STUDYHMW      38425
MISCED        33442
FISCED        45428
ICTRES       262885
HOMEPOS       15730
ESCS          25468
PV1MATH           0
dtype: int64


,CNT,ST004D01T,STUDYHMW,MISCED,FISCED,ICTRES,HOMEPOS,ESCS,PV1MATH
0,ALB,1.0,10.0,7.0,3.0,4.9507,1.5995,1.1112,179.583
1,ALB,2.0,NaN,2.0,3.0,-3.4930,-3.8115,-3.0507,308.247
2,ALB,2.0,0.0,5.0,3.0,0.4307,0.2314,-0.1867,267.514
3,ALB,1.0,10.0,2.0,2.0,-2.1392,-2.5956,-3.2198,272.649
4,ALB,1.0,10.0,5.0,3.0,-0.5542,-0.5632,-1.0548,435.473


## Cleaning the Data

The raw PISA data uses numeric codes for most categorical variables — we need to map those to readable labels. Here's what we're doing:

**Education levels (MISCED / FISCED):** The PISA codes run from 1 (less than primary) to 10 (doctoral). I'm binning these into 7 broader categories to keep the model manageable.

**Books at home (ST013Q01TA):** Coded 1 through 6, mapping to ranges from "0-10" up to "500+".

**Internet access (ST011Q06TA):** 1 = Yes, 2 = No.

**Missing values:** We drop rows missing the target (math_score) or our most important feature (ESCS). For other variables, we fill with "Unknown" so the model can still use the remaining features for that row.


In [4]:
rename_map = {
    'CNT': 'country',
    'ST004D01T': 'gender',
    'MISCED': 'mother_educ_raw',
    'FISCED': 'father_educ_raw',
    'ESCS': 'escs',
    'HOMEPOS': 'home_possessions',
    'ICTRES': 'ict_resources',
    'STUDYHMW': 'study_hours_homework',
    'PV1MATH': 'math_score'
}
# Only rename columns that actually exist
rename_map = {k: v for k, v in rename_map.items() if k in pdf.columns}
pdf = pdf.rename(columns=rename_map)

# Drop rows missing the target or ESCS
initial = len(pdf)
pdf = pdf.dropna(subset=['math_score', 'escs'])
print(f"Dropped {initial - len(pdf):,} rows with missing math_score or ESCS")
print(f"Remaining: {len(pdf):,} rows")

# Map parent education codes to categories
educ_map = {
    1: 'Below_secondary', 2: 'Below_secondary', 3: 'Lower_secondary',
    4: 'Upper_secondary', 5: 'Upper_secondary',
    6: 'Post_secondary', 7: 'Short_cycle_tertiary',
    8: 'Bachelors', 9: 'Masters_plus', 10: 'Masters_plus'
}
if 'mother_educ_raw' in pdf.columns:
    pdf['mother_educ'] = pdf['mother_educ_raw'].map(educ_map).fillna('Unknown')
else:
    pdf['mother_educ'] = 'Unknown'
if 'father_educ_raw' in pdf.columns:
    pdf['father_educ'] = pdf['father_educ_raw'].map(educ_map).fillna('Unknown')
else:
    pdf['father_educ'] = 'Unknown'

# Clean gender
gender_map = {1 : "Female", 2 : "Male", 1.0 : "Female", 2.0 : "Male"}
pdf["gender"] = pdf["gender"].map(gender_map).fillna("Unknown")

# Fill NaNs in numeric columns
for col in ['home_possessions', 'ict_resources', 'study_hours_homework']:
    if col in pdf.columns:
        pdf[col] = pd.to_numeric(pdf[col], errors='coerce').fillna(0)

# Select final columns
final_cols = ['country', 'gender', 'mother_educ', 'father_educ', 'escs',
              'home_possessions', 'ict_resources', 'study_hours_homework', 'math_score']
final_cols = [c for c in final_cols if c in pdf.columns]
pdf_clean = pdf[final_cols].copy()

Dropped 25,468 rows with missing math_score or ESCS
Remaining: 588,276 rows


## Save to CSV

We're saving the cleaned data as a CSV file.

The file should be about **15-25 MB** — dramatically smaller than the 2 GB original because we've cut from 1,200+ columns down to 10 and dropped the SPSS overhead.

In [5]:
OUTPUT_PATH = "pisa_2022_math_clean.csv"
pdf_clean.to_csv(OUTPUT_PATH, index=False)

size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"Saved: {OUTPUT_PATH} ({size_mb:.1f} MB)")
print(f"  {len(pdf_clean):,} rows x {len(pdf_clean.columns)} columns")
print(f"Original data: ~2 GB -> Cleaned CSV: {size_mb:.1f} MB")

Saved: pisa_2022_math_clean.csv (42.6 MB)
  588,276 rows x 9 columns
Original data: ~2 GB -> Cleaned CSV: 42.6 MB


## Done!

**Files Created:**
- `pisa_2022_math_clean.csv`